# QniNotebook：QSCI入力回路の構築確認

QSCI（Quantum-Selected Configuration Interaction）は、VQEなどで得られた近似波動関数を測定し、基底状態を特徴づける重要な電子配置を抽出し、それらのみを用いて小さな有効ハミルトニアンを構成・対角化する量子古典ハイブリッド手法です。本来は指数的に大きい全ての電子配置（ヒルベルト空間）全体を扱う代わりに、解を記述するのに重要な電子配置だけに計算を限定することで、大規模な固有値問題を効率よく解きます。

> このデモでQniNotebookでは、QSCI計算に進む前に、試行状態を作る量子回路が設計どおり組み立てられているかを確認する場面を扱います。

対象は直線に４つ水素原子が並ぶモデル分子H₄です。簡略化した軌道モデル（STO-3G）と電子の状態を量子ビットで表す方法（Jordan–Wigner写像）を使い、8つの電子が入る軌道（スピン軌道）を8量子ビットで表します。ビットの `1` は対応する軌道が占有されていることを示します。H₄は4電子なので、条件に合う測定結果のみの採用（post-selection）で電子数4、上向き・下向き電子数が同数（$S_z=0$） の条件を満たす状態を対象にします。

論文の実験では、電子配置の初期近似（Hartree–Fock状態）から深さ8の $R_y$ 変分回路を作り、勾配を近似する最適化法（BFGS）で最適化した状態を10,000 shots測定します。本デモでQniNotebookを使うのは、**QURI Partsで変分回路を構築した直後、VQEの最適化とサンプリングへ進む前**です。

まず全てのRyゲートの角度を0にした既知の条件で、初期配置、ゲートの接続、途中の状態、最終状態を確認します。これはVQEやQSCIの計算結果を再現するものではなく、論文Appendix C3 / Fig. 10に示された8量子ビット変分回路と、QSCIの入力状態を測定する直前の確認工程を扱うNotebookです。

**参考：**
- [QSCI原論文](https://arxiv.org/abs/2302.11320)
- [QURI SDK公式QSCIチュートリアル](https://quri-sdk.qunasys.com/docs/howto/applying_algorithms/quantum_selected_configuration_interaction)

## このデモの目的

量子アルゴリズムの実装では、回路が長くなるほど、Pythonコードだけから回路全体の構造と各段階の状態変化を把握することが難しくなります。今回例に挙げる状態準備の量子回路では、最終的な計算結果が期待と異なる場合、初期状態、ゲートの接続、パラメーター、測定のどこに原因があるのかを切り分ける必要があります。

量子回路のどの位置で状態が変わったのかを調べ、その結果をほかの人と共有できることが重要です。回路構造と、その位置まで実行した状態を対応付けて確認できれば、問題箇所をより具体的に議論できます。

> そこで、**QURI Partsで構築した量子回路をNotebook上で可視化し、選択した回路位置までの状態を同じ画面で確認できれば、状態準備の意図確認と問題箇所の切り分けをしやすくなる**、という仮説を置きます。

QniNotebookは、この仮説を検証するためのプロトタイプです。**AIによる量子回路のコードレビューを置き換えるものではありません。** AIは実装案、誤りの候補、テスト作成を支援できます。一方、生成された回路が研究者の意図した状態変化を実現するかは、仕様、実行可能なテスト、既知のチェック条件と照合して確認する必要があります。QniNotebookが加えるのは、実際のQURI Parts回路と、その位置までの状態を人間が同じ画面で照合する工程です。

このデモでは、QSCIアルゴリズムの入力状態を準備する回路を例に、次のことを確認します。

- この再現実装で定めたHartree–Fock配置 `00001111` が、意図した量子ビット上に作られているか
- 8回の繰返しが、それぞれ7本の隣接CNOTゲートと8本の $R_y$ ゲートで構成されているか
- 測定直前の状態が `00001111` に戻るか
- スピン軌道と量子ビットの順序を逆に解釈した回路で、別の電子配置になっていることを特定できるか

> ここで確認できるのは、回路と状態を対応付けて表示できることです。レビュー時間や理解度が実際に改善するか、QniNotebookを使わない場合より有用かは、このデモだけでは判断できず、別途利用者評価が必要です。

### このデモで検証する仮説

量子回路のコードは、AIレビュー、静的解析、自動テストでも確認できます。また、IBM Quantum ComposerやMicrosoft QDKなど、回路と状態を視覚的に調べる既存ツールもあります。したがって、このデモは「既存の確認手段では量子回路を検証できない」ことを前提にしていません。

ここで検証したいのは、**QURI Partsで回路を構築しているNotebookから離れず、回路上の位置とその時点までの状態を対応付けて確認できれば、期待状態から外れ始めた箇所を調べやすくなるのではないか**、というワークフロー上の仮説です。Pythonコード、静的な回路図、状態計算を別々に確認する代わりに、同じ画面上で接続、順序、パラメーターと状態変化を照合します。

> **このデモが保証しないこと**  
> 状態の表示だけで回路の正しさや安全性を証明することはできません。期待状態を定義できる箇所では自動テストや、量子プログラムに対するアサーション（期待する状態・性質の検査）を優先し、変換前後の正しさには回路同値性検証を使う必要があります。また現状、状態ベクトル確認は小規模なシミュレーションを対象とし、実機の未知状態を途中で読み取るものではありません。

実在する量子プログラムのバグ研究と、量子プログラムに対するアサーションの研究は、ゲート、初期状態、繰返し、アンコンピューティングなどに確認観点があることを示します。ただし、これらはQniNotebookの必要性を直接証明するものではありません。LLMの量子コード生成ベンチマークも、特定の課題とモデルにおける結果であり、量子プログラミング・AI生成回路一般の危険性を示す根拠としては扱いません。QniNotebookの有用性は、QURI Partsだけを使う場合との比較で、問題箇所の特定時間、操作数、正答率を測って判断します。

**参考資料**

- [Challenges and Practices in Quantum Software Testing and Debugging: Insights from Practitioners](https://arxiv.org/abs/2506.17306)
- [A Comprehensive Study of Bug Fixes in Quantum Programs](https://arxiv.org/abs/2201.08662)
- [Statistical Assertions for Validating Patterns and Finding Bugs in Quantum Programs](https://doi.org/10.1145/3307650.3322213)
- [IBM Quantum Composer](https://quantum.cloud.ibm.com/docs/en/guides/composer)
- [Microsoft QDK：量子回路の可視化](https://learn.microsoft.com/en-us/azure/quantum/how-to-visualize-circuits)


## 1. 論文と同じ8量子ビット変分回路を作る

論文Section IVでは、直線H₄のVQEに深さ8の $R_y$ 変分回路を使っています。Appendix C3 / Fig. 10に示された構造は、Hartree–Fock状態を初期状態とし、最初に8本の $R_y$ ゲートを置いた後、7本の隣接CNOTゲートと8本の $R_y$ ゲートを8回繰り返すものです。すべての回転ゲートが独立したパラメーターを持つため、パラメーター数は $8\times(8+1)=72$ です。

> ※このデモでは、表示結果をあらかじめ予測できるよう、72個の回転角をすべて0にします。これはVQEで得られたパラメーターではなく、回路構築を確認するためのチェック用条件です。

論文Appendix C3に合わせ、量子ビット `q0` から `q3` をXゲートで占有させます。表示上のビット列は `00001111` で、左から `q7 ... q0` の順に並びます。QURI/OpenFermionの交互スピン軌道順序では4電子かつ $S_z=0$ のHartree–Fock配置に対応します。

In [1]:
from qni_jupyter import qni
from quri_parts.circuit import QuantumCircuit

QUBIT_COUNT = 8
DEPTH = 8
ZERO_PARAMETERS = [0.0] * (QUBIT_COUNT * (DEPTH + 1))


def build_h4_ry_ansatz(
    parameters, *, measure=False, occupied_qubits=(0, 1, 2, 3)
):
    if len(parameters) != QUBIT_COUNT * (DEPTH + 1):
        raise ValueError("72 parameters are required")

    if len(occupied_qubits) != 4 or len(set(occupied_qubits)) != 4:
        raise ValueError("occupied_qubits must contain four distinct qubits")
    if any(q < 0 or q >= QUBIT_COUNT for q in occupied_qubits):
        raise ValueError("occupied_qubits contains an out-of-range qubit")

    circuit = QuantumCircuit(
        QUBIT_COUNT, cbit_count=QUBIT_COUNT if measure else 0
    )
    for q in occupied_qubits:
        circuit.add_X_gate(q)  # Hartree–Fock |00001111>

    parameter_index = 0
    for q in range(QUBIT_COUNT):
        circuit.add_RY_gate(q, parameters[parameter_index])
        parameter_index += 1

    for layer in range(DEPTH):
        for q in range(QUBIT_COUNT - 1):
            circuit.add_CNOT_gate(q, q + 1)
        for q in range(QUBIT_COUNT):
            circuit.add_RY_gate(q, parameters[parameter_index])
            parameter_index += 1

    if measure:
        bits = list(range(QUBIT_COUNT))
        circuit.measure(bits, bits)
    return circuit


state_preparation_circuit = build_h4_ry_ansatz(ZERO_PARAMETERS)
assert state_preparation_circuit.qubit_count == 8
assert len(state_preparation_circuit.gates) == 132  # X×4 + ansatz 128
state_preparation_circuit

## 2. 回路構造を確認する

`show_circuit()` で、QURI Partsが生成した回路を表示します。最初の4本のXゲートがHartree–Fock配置を作り、その後に72本の $R_y$ ゲートと56本のCNOTゲートが続きます。各繰返しが、`q0` から `q7` へつながる7本のCNOT ladderと8本の $R_y$ ゲートで構成されていることを確認します。

> ここでは状態の値ではなく、ゲートの種類、順序、接続、繰返し回数を回路の設計と照合します。

### 実行イメージ

<img src="doc/image/qni.show_circuit.png" alt="実行イメージ" width="700">


In [2]:
qni.show_circuit(state_preparation_circuit)

QniViewer(url='http://127.0.0.1:47159/jupyter.html?state=%7B%22steps%22%3A%5B%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B0%5D%7D%2C%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%7D%2C%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B2%5D%7D%2C%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B3%5D%7D%5D%2C%5B%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B0%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B1%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B2%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B3%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B4%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B5%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B6%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B7%5D%2C%22angle%22%3A%220%22%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%2C%22controls%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B2%5D%2C%22controls%22%3A%5B1%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B3%5D%2C%22controls%22%3A%5B2%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B4%5D%2C%22controls%22%3A%5B3%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B5%5D%2C%22controls%22%3A%5B4%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B6%5D%2C%22controls%22%3A%5B5%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B7%5D%2C%22controls%22%3A%5B6%5D%7D%5D%2C%5B%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B0%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B1%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B2%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B3%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B4%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B5%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B6%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B7%5D%2C%22angle%22%3A%220%22%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%2C%22controls%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B2%5D%2C%22controls%22%3A%5B1%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B3%5D%2C%22controls%22%3A%5B2%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B4%5D%2C%22controls%22%3A%5B3%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B5%5D%2C%22controls%22%3A%5B4%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B6%5D%2C%22controls%22%3A%5B5%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B7%5D%2C%22controls%22%3A%5B6%5D%7D%5D%2C%5B%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B0%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B1%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B2%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B3%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B4%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B5%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B6%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B7%5D%2C%22angle%22%3A%220%22%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%2C%22controls%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B2%5D%2C%22controls%22%3A%5B1%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B3%5D%2C%22controls%22%3A%5B2%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B4%5D%2C%22controls%22%3A%5B3%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B5%5D%2C%22controls%22%3A%5B4%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B6%5D%2C%22controls%22%3A%5B5%5D%7D%5D%2

## 3. 最終状態は期待どおりか

この回路で期待している最終状態は `00001111` です。ところが、これから確認する回路には、状態準備の段階に1か所だけ順序の誤りを入れています。132個のゲートを眺めるだけで、その位置を探すのは簡単ではありません。

まず `show_circuit_and_state()` で回路の最後を選び、確率100%のビット列を確認します。期待値 `00001111` に対し、実際には `11110000` が表示されます。どちらも1が4個なので、電子数が4であることだけを確認する検査では、この違いを見逃します。

> **ここでは、まだ原因を決めません。** 最終状態が期待と違うことだけを確認し、次の節で回路を先頭から追って、最初に状態が違った位置を探します。

この検査では、原因を追いやすくするため、72個の回転角をすべて0にしています。そのため、正しく構築された回路の最終状態が `00001111` になることを事前に計算できます。VQEで最適化した回路では複数のビット列が現れるため、この判定基準は使いません。

### 実行イメージ

<img src="doc/image/qni.show_circuit_and_state.png" alt="実行イメージ" width="700">


In [3]:
suspect_circuit = build_h4_ry_ansatz(
    ZERO_PARAMETERS, occupied_qubits=(4, 5, 6, 7)
)
qni.show_circuit_and_state(suspect_circuit)

QniViewer(url='http://127.0.0.1:47159/jupyter.html?state=%7B%22steps%22%3A%5B%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B4%5D%7D%2C%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B5%5D%7D%2C%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B6%5D%7D%2C%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B7%5D%7D%5D%2C%5B%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B0%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B1%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B2%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B3%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B4%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B5%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B6%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B7%5D%2C%22angle%22%3A%220%22%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%2C%22controls%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B2%5D%2C%22controls%22%3A%5B1%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B3%5D%2C%22controls%22%3A%5B2%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B4%5D%2C%22controls%22%3A%5B3%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B5%5D%2C%22controls%22%3A%5B4%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B6%5D%2C%22controls%22%3A%5B5%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B7%5D%2C%22controls%22%3A%5B6%5D%7D%5D%2C%5B%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B0%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B1%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B2%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B3%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B4%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B5%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B6%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B7%5D%2C%22angle%22%3A%220%22%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%2C%22controls%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B2%5D%2C%22controls%22%3A%5B1%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B3%5D%2C%22controls%22%3A%5B2%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B4%5D%2C%22controls%22%3A%5B3%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B5%5D%2C%22controls%22%3A%5B4%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B6%5D%2C%22controls%22%3A%5B5%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B7%5D%2C%22controls%22%3A%5B6%5D%7D%5D%2C%5B%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B0%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B1%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B2%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B3%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B4%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B5%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B6%5D%2C%22angle%22%3A%220%22%7D%2C%7B%22type%22%3A%22Ry%22%2C%22targets%22%3A%5B7%5D%2C%22angle%22%3A%220%22%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B1%5D%2C%22controls%22%3A%5B0%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B2%5D%2C%22controls%22%3A%5B1%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B3%5D%2C%22controls%22%3A%5B2%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B4%5D%2C%22controls%22%3A%5B3%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B5%5D%2C%22controls%22%3A%5B4%5D%7D%5D%2C%5B%7B%22type%22%3A%22X%22%2C%22targets%22%3A%5B6%5D%2C%22controls%22%3A%5B5%5D%7D%5D%2

### 判定基準

| 選択位置 | 期待する表示 | 確認内容 |
|:---|:---|:---|
| 回路の最後 | `00001111`：100% | 実際は `11110000`：100%となり、不一致を発見 |

> この基準は、回転角をすべて0にしたチェック用条件にだけ適用します。VQEで最適化したパラメーターでは複数配置の重ね合わせになるため、最終状態が同じビット列である必要はありません。

## 4. どこから状態が違っているかを探す

最終状態が違うことは分かりました。次に、回路の選択位置を先頭へ戻し、状態が最初に作られる位置を確認します。

この実装の仕様は「q0〜q3をXゲートで反転し、画面ではq7からq0の順で読む」です。したがって、最初の4本のXゲートを通過した直後には `00001111` が100%になるはずです。

しかし、問題の回路ではXゲートがq4〜q7に置かれ、その直後から `11110000` になっています。つまり、後続の $R_y$ ゲートやCNOTゲートではなく、**状態準備の時点が最初の不一致**です。

> **画面での探し方**：左端にある4本のXゲートの直後を選択します。次に、Xゲートがq0〜q3とq4〜q7のどちらにあるか、状態欄の100%のビット列が `00001111` と `11110000` のどちらかを見比べます。

| 選択位置 | 期待 | 実際 | 判断 |
|:---|:---|:---|:---|
| Xゲートの前 | `00000000` | `00000000` | ここまでは一致 |
| 4本のXゲートの直後 | `00001111` | `11110000` | **ここが最初の不一致** |
| 回路の最後 | `00001111` | `11110000` | 誤った初期配置が最後まで残っている |

この誤りは、単にXゲートを置き間違えたという設定ではありません。量子化学側の軌道番号、回路の量子ビット番号、画面のビット列が逆順で表記されることを、連携部分で取り違えた状況を単純化しています。QniNotebookは正しい規約を自動決定しませんが、仕様上の期待値と実回路の状態を位置ごとに照合できます。


In [ ]:
reference_circuit = build_h4_ry_ansatz(ZERO_PARAMETERS)
assert len(suspect_circuit.gates) == len(reference_circuit.gates)
qni.show_circuit_and_state(reference_circuit)

## 5. 測定を含む回路を確認する

最後に、8量子ビットを計算基底で測定する回路を表示します。全回転角を0にしたチェック用条件では、測定結果は `00001111` になります。8量子ビットが同じ番号の古典bitへ対応付けられていることも確認します。

> 論文の10,000 shotsによるサンプリング、頻度上位 $R=1,4,16,27$ の配置選択、$N_e=4, S_z=0$ によるpost-selection、部分空間ハミルトニアンの古典対角化は `qsci_h4_paper_reproduction.ipynb` で扱います。このNotebookの状態表示や1回の測定結果は、論文のQSCIサンプリング結果そのものではありません。

### 実行イメージ

**右にスクロールし、測定結果を確認**

<img src="doc/image/qni.show_circuit(sampling_circuit).png" alt="実行イメージ" width="700">


In [ ]:
sampling_circuit = build_h4_ry_ansatz(ZERO_PARAMETERS, measure=True)
qni.show_circuit(sampling_circuit, scroll_to="last")

## 6. このデモで確認できること

このデモでは、QSCI原論文Section IVのH₄実験で使われた8量子ビット・depth 8の $R_y$ ansatzをQURI Partsで構築し、VQEの最適化とサンプリングへ進む前に、その構造と状態を確認しました。全回転角を0にしたチェック用条件ではHartree–Fock配置に戻ることを、回路上の位置と状態ベクトルを対応付けて確認できます。

### ワークフローとの関係

例としてあげた論文のQSCIでは、

- (1) VQEなどで入力状態を準備
- (2) 計算基底で測定
- (3) 保存量でpost-selectionを行う
- (4) 頻度の高い配置を選ぶ
- (5) 選択した部分空間のハミルトニアンを古典対角化する

という流れです。このNotebookが扱うのは、そのうち(1)の回路を構築した直後です。まず全パラメーター0などの既知条件で構造と状態遷移を確認し、その後にVQE最適化へ進みます。実用化する場合は、最適化後・サンプリング前にも数値確定済み回路を再表示し、量子ビット数、ゲート接続、パラメーター束縛、測定対応を確認します。QSCI自体を変更せず、計算コストの大きい最適化・サンプリングの境界に確認工程を置くため、研究フローへの挿入に大きな無理はありません。

論文との対応は、8量子ビット、4電子、Hartree–Fock初期状態、depth 8の $R_y$ ansatz、独立した回転パラメーター、隣接CNOT ladderです。一方、BFGSによる最適化、10,000-shotの分布、device noise、QSCIの配置選択とエネルギー計算は再現していません。そのため、これはFig. 8の結果再現ではなく、結果を得る前の入力回路を確認するデモです。

### 対応状況

| 項目 | 現状 | 根拠／残課題 |
|:---|:---:|:---|
| 課題起点のシナリオ | 対応 | 回路構造の確認と、誤り位置の特定に整理。GUI編集は初期デモから除外 |
| 4〜8量子ビットの現実的な回路 | 対応 | H₄の8量子ビット・depth 8・132ゲートを使用 |
| 既知の誤りから最初の不一致を探す | 今回対応 | スピン軌道・量子ビット・表示ビット列の順序不一致を、同じ電子数の別配置で再現 |
| 回路と表示状態の対応 | 対応 | 回路内容を含むキャッシュキー、非同期応答の世代管理、ブラウザE2Eで検証 |
| 未対応操作の安全な拒否 | 対応 | 未対応ゲート、アンチコントロール、保持できない測定対応は表示を停止 |
| 実行範囲と再現環境 | 対応 | 1〜8量子ビット、256 KiB、10秒上限、Docker ComposeとCIを用意 |
| QSCI最適化済み回路との接続 | 未対応 | このNotebookは全パラメーター0。BFGS結果を束縛した回路による確認が次段階 |
| 利用者価値の実証 | 未対応 | Qniあり／なしで発見率と特定時間を比較する必要がある |

### このデモでまだ分からないこと

> QniNotebookを使うことでレビュー時間や理解度が実際に改善するか、また通常の確認方法より有用かは、このデモだけでは判断できません。回路の誤りを見つける作業をQniNotebookあり／なしで比較し、発見率、原因箇所の特定時間、説明に必要なやり取りを利用者評価で調べる必要があります。

> このデモは、8量子ビットまでの完全状態ベクトルによる回路構築確認です。QSCIによる配置選択やエネルギー計算は行いません。

### 今後の拡張順序

1. **現在の主導線を検証する**：QunaSysの技術者に、スピン軌道と量子ビットの順序が異なる回路でQniあり／なしの問題特定を試してもらう。
2. **QSCIとの接続を完成する**：最適化済みパラメーターを束縛したQURI Parts回路を、サンプリング直前に表示する。状態ベクトル全体の目視ではなく、期待する対称性sectorの確率、主要配置、回路fingerprintを数値で併記する。
3. **チェックポイントを部品境界へ拡張する**：Qsubやcompute–uncomputeの境界に名前付きチェックポイントを置き、補助量子ビットが $|0\rangle$ に戻る等の事前・事後条件を確認できるようにする。これはuncomputationを含む回路には有効だが、今回のH₄ $R_y$ ansatzにuncomputationはないため、このデモへ無理に入れない。
4. **大規模回路は別方式にする**：8量子ビット超では完全状態ベクトルを標準にせず、部分回路、選択した振幅、保存量、サンプリング分布へ切り替える。
5. **GUI編集は価値検証後に再評価する**：未反映表示、Apply操作、差分、Notebook再実行時の世代管理を設計できるまで、研究デモの主導線には戻さない。

> 既出の『Qniで操作してQURIコードを学ぶ』案は教育用途としては残します。ただし、今回の研究者向け提案とは対象ユーザーと評価指標が異なるため、同じ最小デモには混ぜません。

## 7. QniNotebookの利用範囲

このデモでは、8量子ビットまでの回路を読み取り専用で表示します。状態表示には完全状態ベクトル simulationを使うため、大規模回路や実機での性能を扱うものではありません。

回路の編集、Pythonコードへの書き戻し、VQEの最適化、10,000-shotのサンプリング、QSCIの配置選択や古典対角化は、このNotebookの対象外です。QniNotebookが解釈できない回路を受け取った場合は、別の意味の回路として表示せず、理由を示して処理を停止します。

以上が、このデモで確認できる機能と、論文のQSCI計算に引き渡す前の位置づけです。